# **Applicazione massiva di FEEL-IT**

> Si tratta di un notebook ausiliario, a cui si ricorre durante l'esecuzione di `3_nlp_sentiment__recensioni.ipynb` per l'applicazione massiva di FEEL-IT sull'ambiente Google Colab, così da sfruttare la GPU che mette a disposizione.

In [1]:
!pip install transformers torch pandas sentencepiece -q

import os, time
import torch
import numpy as np
import pandas as pd
from google.colab import files
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

In [2]:
device = 0 if torch.cuda.is_available() else -1
if device == -1:
    raise RuntimeError("GPU non attiva. Runtime → Cambia tipo di runtime → GPU (T4).")
print(f"GPU: {torch.cuda.get_device_name(0)}\n")

df = pd.read_csv("3_1_input_per_colab.csv", index_col="ID Recensione")
df_valido = df[
    df["Corpus_Sentiment"].notna() &
    (df["Corpus_Sentiment"].astype(str).str.strip() != "")
].copy()

testi = df_valido["Corpus_Sentiment"].astype(str).tolist()
print(f"Righe da analizzare: {len(testi):,}")

GPU: Tesla T4

Righe da analizzare: 97,240


**Nota sul caricamento del modello.** Le versioni recenti della libreria `transformers` presentano un'incompatibilità con il tokenizer CamemBERT usato da UmBERTo (il vocabolario viene passato come dizionario, mentre il costruttore attende una lista di tuple). Si adotta un doppio livello di difesa: si tenta prima il caricamento standard dall'hub di Hugging Face; solo in caso di fallimento si ricorre al download manuale dei file e a un patch puntuale del costruttore del tokenizer.

In [ ]:
CARTELLA = "/content/feelit"
MODELLO_HF = "MilaNLProc/feel-it-italian-sentiment"

# Tentativo 1: caricamento diretto dall'hub, senza workaround
def carica_pipeline_standard():
    return pipeline("text-classification", model=MODELLO_HF, device=device, top_k=None, truncation=True, max_length=512)

# Tentativo 2: download manuale + patch del tokenizer CamemBERT
def carica_pipeline_con_patch():
    BASE = f"https://huggingface.co/{MODELLO_HF}/resolve/main"
    os.system(f"rm -rf {CARTELLA} && mkdir -p {CARTELLA}")

    # Si scaricano tutti i file potenzialmente necessari. Alcuni potrebbero non esistere nel repo: in tal caso wget crea un file vuoto, scartato dopo.
    for f in ["config.json", "pytorch_model.bin", "sentencepiece.bpe.model",
              "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
              "vocab.json", "merges.txt"]:
        os.system(f"wget -q -O {CARTELLA}/{f} {BASE}/{f}")
    for f in os.listdir(CARTELLA):
        p = os.path.join(CARTELLA, f)
        if os.path.getsize(p) < 50:
            os.remove(p)
    print("File validi scaricati:", sorted(os.listdir(CARTELLA)))

    # Bug noto di transformers: CamembertTokenizer riceve il vocabolario come dict, ma il costruttore si aspetta una lista di tuple (token, score). Il patch converte il formato prima di passarlo al costruttore originale.
    from transformers.models.camembert.tokenization_camembert import CamembertTokenizer
    if not hasattr(CamembertTokenizer, "_patched"):
        _orig_init = CamembertTokenizer.__init__
        def _patched_init(self, *args, **kwargs):
            v = kwargs.get("vocab")
            if isinstance(v, dict):
                kwargs["vocab"] = [(k, 0.0) for k in v.keys()]
            _orig_init(self, *args, **kwargs)
        CamembertTokenizer.__init__ = _patched_init
        CamembertTokenizer._patched = True
        print("Patch applicata al tokenizer CamemBERT.")

    modello   = AutoModelForSequenceClassification.from_pretrained(CARTELLA)
    tokenizer = AutoTokenizer.from_pretrained(CARTELLA)
    return pipeline("text-classification", model=modello, tokenizer=tokenizer, device=device, top_k=None, truncation=True, max_length=512)

try:
    feelit_pipeline = carica_pipeline_standard()
    print("Caricamento standard riuscito (nessun workaround necessario).")
except Exception as e:
    print(f"Caricamento standard fallito ({type(e).__name__}): si passa al workaround.\n")
    feelit_pipeline = carica_pipeline_con_patch()

print("\nPipeline pronta. Test:")
prova = ["Ottimo prodotto, lo consiglio", "Pessimo, si è rotto subito"]
for testo, out in zip(prova, feelit_pipeline(prova)):
    print(f"  '{testo}' → {({r['label'].lower(): round(r['score'], 3) for r in out})}")

config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.68M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/299 [00:00<?, ?B/s]

Caricamento standard fallito (ValueError): si passa al workaround.

File validi scaricati: ['config.json', 'pytorch_model.bin', 'sentencepiece.bpe.model', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']
Patch applicata al tokenizer CamemBERT.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Pipeline pronta. Test:
  'Ottimo prodotto, lo consiglio' → {'positive': 1.0, 'negative': 0.0}
  'Pessimo, si è rotto subito' → {'negative': 1.0, 'positive': 0.0}


In [ ]:
print("\nAvvio inferenza...\n", flush=True)
BATCH = 64

# Ordinamento per lunghezza: batch omogenei, molto meno padding sprecato
ordine = sorted(range(len(testi)), key=lambda i: len(testi[i]))
testi_ord = [testi[i] for i in ordine]

n_batch = int(np.ceil(len(testi) / BATCH))
pos_ord, neg_ord = [], []
t0 = time.time()

for k, i in enumerate(range(0, len(testi_ord), BATCH)):
    for out in feelit_pipeline(testi_ord[i:i + BATCH]):
        prob = {r["label"].lower(): r["score"] for r in out}
        pos_ord.append(prob.get("positive", 0.0))
        neg_ord.append(prob.get("negative", 0.0))

    if k % 100 == 0:
        el = time.time() - t0
        fatti = len(pos_ord)
        eta = (el / max(fatti, 1)) * (len(testi) - fatti) / 60
        print(f"  batch {k:,}/{n_batch:,} | {fatti:,} ({fatti/len(testi)*100:.1f}%) | ETA {eta:.1f} min", flush=True)

print(f"\nCompletato in {(time.time()-t0)/60:.1f} minuti.", flush=True)

# Ripristino ordine originale
n = len(testi)
p_pos, p_neg = [0.0]*n, [0.0]*n
for pos, idx in enumerate(ordine):
    p_pos[idx] = pos_ord[pos]
    p_neg[idx] = neg_ord[pos]

df_valido["P_pos_FEELIT"] = p_pos
df_valido["P_neg_FEELIT"] = p_neg

# Score atteso su due classi: E[S] = P(pos) - P(neg)
df_valido["Sentiment_FEELIT"] = (df_valido["P_pos_FEELIT"] - df_valido["P_neg_FEELIT"]).round(4)
print("Score calcolati.")


Avvio inferenza...

  batch 0/1,520 | 64 (0.1%) | ETA 14.0 min


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  batch 100/1,520 | 6,464 (6.6%) | ETA 15.5 min
  batch 200/1,520 | 12,864 (13.2%) | ETA 15.4 min
  batch 300/1,520 | 19,264 (19.8%) | ETA 14.0 min
  batch 400/1,520 | 25,664 (26.4%) | ETA 12.5 min
  batch 500/1,520 | 32,064 (33.0%) | ETA 11.3 min
  batch 600/1,520 | 38,464 (39.6%) | ETA 10.0 min
  batch 700/1,520 | 44,864 (46.1%) | ETA 8.8 min
  batch 800/1,520 | 51,264 (52.7%) | ETA 7.7 min
  batch 900/1,520 | 57,664 (59.3%) | ETA 6.6 min
  batch 1,000/1,520 | 64,064 (65.9%) | ETA 5.5 min
  batch 1,100/1,520 | 70,464 (72.5%) | ETA 4.4 min
  batch 1,200/1,520 | 76,864 (79.0%) | ETA 3.3 min
  batch 1,300/1,520 | 83,264 (85.6%) | ETA 2.3 min
  batch 1,400/1,520 | 89,664 (92.2%) | ETA 1.2 min
  batch 1,500/1,520 | 96,064 (98.8%) | ETA 0.2 min

Completato in 15.9 minuti.
Score calcolati.


In [5]:
NOME = "3_3_feel_it.csv"
df_valido[["Sentiment_FEELIT", "P_pos_FEELIT", "P_neg_FEELIT"]].to_csv(NOME)

s = df_valido["Sentiment_FEELIT"]
print(f"Salvato: {NOME} ({len(df_valido):,} righe)\n")
print(f"Media: {s.mean():.3f} | Mediana: {s.median():.3f}")
print(f"Positivi (> 0.05):  {(s >  0.05).sum():,}")
print(f"Neutri:             {((s >= -0.05) & (s <= 0.05)).sum():,}")
print(f"Negativi (< -0.05): {(s < -0.05).sum():,}")

# FEEL-IT ha solo due classi: ci si attende una distribuzione fortemente bimodale.
estremi   = (s.abs() > 0.9).sum()
intermedi = ((s.abs() >= 0.1) & (s.abs() <= 0.9)).sum()
zero      = (s.abs() < 0.1).sum()

print(f"\nPolarizzazione:")
print(f"  estremi   (|s| > 0.9): {estremi:,} ({estremi/len(s)*100:.1f}%)")
print(f"  intermedi:             {intermedi:,} ({intermedi/len(s)*100:.1f}%)")
print(f"  quasi zero:            {zero:,} ({zero/len(s)*100:.1f}%)")

files.download(NOME)

Salvato: 3_3_feel_it.csv (97,240 righe)

Media: 0.712 | Mediana: 1.000
Positivi (> 0.05):  83,256
Neutri:             70
Negativi (< -0.05): 13,914

Polarizzazione:
  estremi   (|s| > 0.9): 94,596 (97.3%)
  intermedi:             2,508 (2.6%)
  quasi zero:            136 (0.1%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>